# -Libraries-


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

ModuleNotFoundError: No module named 'numpy'

# -Cleaning-


In [ ]:

# Load datasets
df = pd.read_csv("PE_dataset/dataset_malwares.csv")
df_test = pd.read_csv("PE_dataset/dataset_test.csv")

# Drop 'Name' column if present
for dataset in [df, df_test]:
    if 'Name' in dataset.columns:
        dataset.drop('Name', axis=1, inplace=True)

# Remove duplicates and missing values
for dataset in [df, df_test]:
    dataset.drop_duplicates(inplace=True)
    dataset.dropna(inplace=True)
    dataset.reset_index(drop=True, inplace=True)

# Quick checks
print("Train shape:", df.shape)
print("Test shape :", df_test.shape)
print("Train head:\n", df.head())
print("Test head:\n", df_test.head())

# Save cleaned data for preprocessing
df.to_csv("PE_dataset/cleaned_train.csv", index=False)
df_test.to_csv("PE_dataset/cleaned_test.csv", index=False)


Train shape: (16631, 78)
Test shape : (17, 77)
Train head:
    e_magic  e_cblp  e_cp  e_crlc  e_cparhdr  e_minalloc  e_maxalloc  e_ss  \
0    23117     144     3       0          4           0       65535     0   
1    23117     144     3       0          4           0       65535     0   
2    23117     144     3       0          4           0       65535     0   
3    23117     144     3       0          4           0       65535     0   
4    23117     144     3       0          4           0       65535     0   

   e_sp  e_csum  ...  SectionMaxChar  SectionMainChar  DirectoryEntryImport  \
0   184       0  ...      3758096608                0                     7   
1   184       0  ...      3791650880                0                    16   
2   184       0  ...      3221225536                0                     6   
3   184       0  ...      3224371328                0                     8   
4   184       0  ...      3227516992                0                     2   

  

# -Preprocessing-


In [ ]:
df = pd.read_csv("PE_dataset/cleaned_train.csv")
df_test = pd.read_csv("PE_dataset/cleaned_test.csv")

target_column = "Malware"

X = df.drop(columns=[target_column])
y = df[target_column]
X_final_test = df_test.drop(columns=[target_column], errors="ignore")

# Encode categorical features
label_encoders = {}
for col in X.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

for col in X_final_test.select_dtypes(include=["object"]).columns:
    if col in label_encoders:
        X_final_test[col] = X_final_test[col].map(
            lambda x: label_encoders[col].transform([x])[0]
            if x in label_encoders[col].classes_ else -1
        )

# ✅ FIX: Split ก่อน แล้วค่อย fit scaler เฉพาะ X_train
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val),       columns=X_val.columns)
X_final_test_scaled = pd.DataFrame(scaler.transform(X_final_test), columns=X_final_test.columns)

joblib.dump(scaler, "Model/scaler.pkl")
joblib.dump(label_encoders, "Model/encoders.pkl")

print("Train shape:", X_train_scaled.shape)
print("Val shape  :", X_val_scaled.shape)
print("Test shape :", X_final_test_scaled.shape)
print("Class distribution:\n", y_train.value_counts())

Train shape: (16631, 77)
Test shape : (17, 77)
Class distribution in train set:
 Malware
1    11665
0     4966
Name: count, dtype: int64


# -Training-


In [ ]:
# ✅ FIX: เพิ่ม class_weight='balanced'
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train_scaled, y_train)

y_val_pred = rf_model.predict(X_val_scaled)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_val_pred))

test_preds = rf_model.predict(X_final_test_scaled)
print("\nFinal test predictions:", test_preds)

joblib.dump(rf_model, "Model/rf_model.pkl")
print("\nModel saved at 'Model/rf_model.pkl'")

Validation Accuracy: 0.9897805831079051

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.98      0.98       993
           1       0.99      1.00      0.99      2334

    accuracy                           0.99      3327
   macro avg       0.99      0.99      0.99      3327
weighted avg       0.99      0.99      0.99      3327


Confusion Matrix:
 [[ 969   24]
 [  10 2324]]
Final test predictions (first 20): [0 0 1 0 0 1 0 1 1 0 1 1 1 1 1 1 0]
RandomForest model saved at 'Model/rf_model.pkl'


In [ ]:
feat_imp = pd.Series(rf_model.feature_importances_, index=X_train_scaled.columns)
top10 = feat_imp.nlargest(10)

plt.figure(figsize=(10, 6))
top10.sort_values().plot(kind='barh', color='steelblue')
plt.title("Top 10 Most Important Features", fontsize=14)
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

print("\nTop 10 Features:")
print(top10.to_string())

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malware'],
            yticklabels=['Benign', 'Malware'])
plt.title('Confusion Matrix', fontsize=14)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()